# NPPE-1: Multilingual Sentiment Analysis
## Base Model: `google/gemma-3-1b-it` | Technique: QLoRA Fine-tuning

### Approach Overview
- **Model**: Google Gemma-3-1B-IT (mandatory base model)
- **Fine-tuning**: QLoRA (4-bit NF4 quantization + LoRA adapters)
- **Task**: Binary sentiment classification — `Positive` / `Negative`
- **Languages**: 13 Indian languages (as, bd, bn, gu, hi, kn, ml, mr, or, pa, ta, te, ur)
- **Metric**: Macro F1-Score

### Pipeline
1. Install & import dependencies
2. Authenticate with HuggingFace (Gemma is gated)
3. Load & explore competition data
4. Build instruction-style prompts with the Gemma chat template
5. Load model in 4-bit and apply LoRA adapters
6. Fine-tune with SFTTrainer (completion-only loss)
7. Evaluate on held-out validation set
8. Run inference on test set and export `submission.csv`

## Step 1: Install Dependencies

In [ ]:
import importlib, subprocess, sys

def _version_tuple(v):
    return tuple(int(x) for x in v.split(".")[:3] if x.isdigit())

REQUIREMENTS = {
    "transformers": ("4.47", "transformers>=4.47"),
    "peft":         ("0.13", "peft>=0.13"),
    "bitsandbytes": ("0.43", "bitsandbytes>=0.43"),
    "trl":          ("0.12", "trl>=0.12"),
    "accelerate":   ("0.27", "accelerate>=0.27"),
    "datasets":     ("2.18", "datasets>=2.18"),
}

needs_install = []
for pkg, (min_ver, spec) in REQUIREMENTS.items():
    try:
        mod = importlib.import_module(pkg)
        installed = getattr(mod, "__version__", "0.0.0")
        if _version_tuple(installed) >= _version_tuple(min_ver):
            print(f"  ✓ {pkg} {installed} (>= {min_ver} required)")
        else:
            print(f"  ✗ {pkg} {installed} — needs upgrade to {min_ver}+")
            needs_install.append(spec)
    except ImportError:
        print(f"  ✗ {pkg} — not installed")
        needs_install.append(spec)

if needs_install:
    print(f"\nInstalling: {needs_install}")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U"
    ] + needs_install)
    print("Installation complete. Please restart the kernel if prompted.")
else:
    print("\nAll packages already satisfy version requirements — no install needed.")


  ✓ transformers 5.2.0 (>= 4.47 required)
  ✓ peft 0.18.1 (>= 0.13 required)
  ✗ bitsandbytes — not installed
  ✗ trl — not installed
  ✓ accelerate 1.12.0 (>= 0.27 required)
  ✓ datasets 4.0.0 (>= 2.18 required)

Installing: ['bitsandbytes>=0.43', 'trl>=0.12']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.7 MB/s eta 0:00:00
Installation complete. Please restart the kernel if prompted.


## Step 2: Imports & Reproducibility

In [2]:

import os
# ── MUST be set before any CUDA/torch import ───────────────────────────────────
# Forces the entire session to use only GPU 0.
# QLoRA + DataParallel across multiple GPUs causes illegal memory access errors.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import glob
import json
import random
import importlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
    PeftModel,
)

# ── TRL: define safe defaults FIRST, then try to import ───────────────────────
SFTTrainer                      = None
DataCollatorForCompletionOnlyLM = None
USE_SFT_CONFIG                  = False

try:
    from trl import SFTTrainer
    print("SFTTrainer imported from trl.")
except ImportError as e:
    raise ImportError(
        "Could not import SFTTrainer from trl. "
        "Please run the pip install cell (cell 3) first, then restart the kernel.\n"
        f"Original error: {e}"
    )

# DataCollatorForCompletionOnlyLM was relocated in TRL >= 0.13 — search all locations
for _mod in ["trl", "trl.trainer", "trl.trainer.utils", "trl.data_utils"]:
    try:
        _m = importlib.import_module(_mod)
        _cls = getattr(_m, "DataCollatorForCompletionOnlyLM", None)
        if _cls is not None:
            DataCollatorForCompletionOnlyLM = _cls
            print(f"DataCollatorForCompletionOnlyLM imported from: {_mod}")
            break
    except Exception:
        continue

if DataCollatorForCompletionOnlyLM is None:
    print("WARNING: DataCollatorForCompletionOnlyLM not found in any trl sub-module.")
    print("         Training will use the standard collator (full-sequence loss) — still works.")

try:
    from trl import SFTConfig
    USE_SFT_CONFIG = True
    print("SFTConfig available.")
except ImportError:
    USE_SFT_CONFIG = False
    print("SFTConfig not available — will use TrainingArguments.")

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # Guard: if the CUDA context is corrupted (e.g. after a crashed training run),
    # manual_seed_all will raise. Skip it — it's non-fatal for reproducibility.
    if torch.cuda.is_available():
        try:
            torch.cuda.manual_seed_all(seed)
        except Exception as _e:
            print(f"WARNING: Could not seed CUDA ({_e}).")
            print("         If you see this, STOP the Kaggle session fully and restart.")

set_seed(SEED)

# ── Device info ────────────────────────────────────────────────────────────────
print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"GPUs visible : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU      : {gpu.name}")
    print(f"VRAM     : {gpu.total_memory / 1e9:.1f} GB")


SFTTrainer imported from trl.
         Training will use the standard collator (full-sequence loss) — still works.
SFTConfig available.

PyTorch  : 2.9.0+cu126
CUDA     : True
GPUs visible : 1
GPU      : Tesla T4
VRAM     : 15.6 GB


## Step 3: HuggingFace Authentication

> **Important**: Gemma-3 is a gated model. You must:
> 1. Accept the Gemma licence at https://huggingface.co/google/gemma-3-1b-it
> 2. In Kaggle → Add-ons → Secrets, add your HF token as `HUGGINGFACE_TOKEN`

In [3]:
HF_TOKEN = ""

# Try Kaggle Secrets first (recommended)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    # Fallback to environment variable
    HF_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", "")
    if HF_TOKEN:
        print("HF token loaded from environment variable.")
    else:
        print("WARNING: No HF token found. Model download may fail for gated models.")
        print("Add your token in Kaggle Secrets as 'HUGGINGFACE_TOKEN'.")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")

HF token loaded from Kaggle Secrets.
Logged in to HuggingFace Hub.


## Step 4: Configuration (All Hyperparameters in One Place)

In [ ]:

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_ID    = "google/gemma-3-1b-it"   # MANDATORY — do NOT change
OUTPUT_DIR  = "/kaggle/working/gemma3-sentiment"
SUBMIT_PATH = "/kaggle/working/submission.csv"

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

# ── LoRA ───────────────────────────────────────────────────────────────────────
LORA_R               = 32       # Increased from 16 → more capacity across 13 languages
LORA_ALPHA           = 64       # Always 2 × r
LORA_DROPOUT         = 0.05
LORA_TARGET_MODULES  = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Training ───────────────────────────────────────────────────────────────────
NUM_EPOCHS          = 6         # Increased from 4 → better convergence on small dataset
LEARNING_RATE       = 1e-4      # Lowered from 2e-4 → finer convergence with more epochs
BATCH_SIZE          = 4
GRAD_ACCUM          = 4         # Effective batch = 16
MAX_SEQ_LEN         = 256
WARMUP_RATIO        = 0.1
WEIGHT_DECAY        = 0.01
TRAIN_ON_FULL_DATA  = True     
VAL_SPLIT           = 0.15      

# ── Inference ──────────────────────────────────────────────────────────────────
INFER_BATCH_SIZE    = 8
MAX_NEW_TOKENS      = 10

# ── Label encoding ─────────────────────────────────────────────────────────────
LABEL2INT = {"Positive": 1, "Negative": 0}
INT2LABEL = {1: "Positive", 0: "Negative"}

# ── Language mapping ───────────────────────────────────────────────────────────
LANGUAGE_MAP = {
    "as": "Assamese", "bd": "Bodo",    "bn": "Bengali",
    "gu": "Gujarati",  "hi": "Hindi",   "kn": "Kannada",
    "ml": "Malayalam", "mr": "Marathi", "or": "Odia",
    "pa": "Punjabi",   "ta": "Tamil",   "te": "Telugu",
    "ur": "Urdu",
}

print("Configuration loaded.")
print(f"  Model            : {MODEL_ID}")
print(f"  SEED             : {SEED}")
print(f"  LoRA rank/alpha  : {LORA_R} / {LORA_ALPHA}")
print(f"  Effective batch  : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Epochs           : {NUM_EPOCHS}")
print(f"  Learning rate    : {LEARNING_RATE}")
print(f"  Max seq length   : {MAX_SEQ_LEN}")
print(f"  Train on full    : {TRAIN_ON_FULL_DATA}  ({'ALL 900 samples' if TRAIN_ON_FULL_DATA else f'{int((1-VAL_SPLIT)*100)}% split'})")
print(f"  Label encoding   : {LABEL2INT}")


Configuration loaded.
  Model            : google/gemma-3-1b-it
  SEED             : 42
  LoRA rank/alpha  : 32 / 64
  Effective batch  : 16
  Epochs           : 6
  Learning rate    : 0.0001
  Max seq length   : 256
  Train on full    : True  (ALL 900 samples)
  Label encoding   : {'Positive': 1, 'Negative': 0}


## Step 5: Data Loading 

In [ ]:

print("Scanning /kaggle/input for data files:")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(f"  {os.path.join(root, f)}")

train_candidates  = sorted(glob.glob("/kaggle/input/**/train.csv",  recursive=True))
test_candidates   = sorted(glob.glob("/kaggle/input/**/test.csv",   recursive=True))
sample_candidates = sorted(glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True))

if not train_candidates:
    raise FileNotFoundError(
        "train.csv not found. Please add the competition dataset via Data → Add Data."
    )

TRAIN_PATH  = train_candidates[0]
TEST_PATH   = test_candidates[0]
SAMPLE_PATH = sample_candidates[0] if sample_candidates else None

print(f"\nUsing train            : {TRAIN_PATH}")
print(f"Using test             : {TEST_PATH}")
print(f"Using sample_submission: {SAMPLE_PATH}")

train_df  = pd.read_csv(TRAIN_PATH)
test_df   = pd.read_csv(TEST_PATH)


if SAMPLE_PATH:
    sample_df = pd.read_csv(SAMPLE_PATH)
    print(f"\nSample submission labels (unique): {sorted(sample_df['label'].unique().tolist())}")
    print(f"Sample submission head:\n{sample_df.head(3).to_string(index=False)}")
 
    test_df = test_df.set_index("ID").reindex(sample_df["ID"].values).reset_index()
    print(f"\nTest re-ordered to match sample_submission ID order: {len(test_df)} rows")
else:
    print("WARNING: sample_submission.csv not found; using test.csv row order.")

print(f"\nTrain shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
print(f"\nTrain columns: {train_df.columns.tolist()}")
print(f"Test  columns : {test_df.columns.tolist()}")


Scanning /kaggle/input for data files:
  /kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv
  /kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
  /kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv

Using train            : /kaggle/input/competitions/nppe-dlp-2026-term-1/train.csv
Using test             : /kaggle/input/competitions/nppe-dlp-2026-term-1/test.csv
Using sample_submission: /kaggle/input/competitions/nppe-dlp-2026-term-1/sample_submission.csv

Sample submission labels (unique): [1]
Sample submission head:
 ID  label
550      1
397      1
757      1

Test re-ordered to match sample_submission ID order: 100 rows

Train shape : (900, 4)
Test  shape : (100, 3)

Train columns: ['ID', 'sentence', 'label', 'language']
Test  columns : ['ID', 'sentence', 'language']


In [ ]:
print("=== Label Distribution ===")
print(train_df["label"].value_counts())

print("\n=== Language Distribution (train) ===")
print(train_df["language"].value_counts())

print("\n=== Language Distribution (test) ===")
print(test_df["language"].value_counts())

print("\n=== Sample lengths (characters) ===")
train_df["char_len"] = train_df["sentence"].str.len()
print(train_df["char_len"].describe().round(1))

print("\n=== Sample rows ===")
display(train_df.head(5))

=== Label Distribution ===
label
Positive    456
Negative    444
Name: count, dtype: int64

=== Language Distribution (train) ===
language
ta    76
hi    74
or    72
pa    72
as    71
bd    71
gu    69
ml    68
mr    67
kn    66
ur    66
bn    65
te    63
Name: count, dtype: int64

=== Language Distribution (test) ===
language
te    14
bn    12
ur    11
kn    11
mr    10
ml     8
gu     8
as     6
bd     6
pa     5
or     5
hi     3
ta     1
Name: count, dtype: int64

=== Sample lengths (characters) ===
count    900.0
mean     136.6
std       73.1
min       14.0
25%       85.0
50%      126.0
75%      169.0
max      527.0
Name: char_len, dtype: float64

=== Sample rows ===


,ID,sentence,label,language,char_len
0,82,ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀ...,Negative,pa,149
1,618,"ਇੱਕ ਸਮਗਰੀ ਦੇ ਰੂਪ ਵਿੱਚ, ਕੋਟਿੰਗ ਤੋਂ ਬਿਨਾਂ, ਸਿਰਫ ...",Negative,pa,79
2,812,"ബ്രിസിലുകൾ കട്ടിയുള്ള പ്ലാസ്റ്റിക് ആണ്, അതിനാൽ...",Negative,ml,78
3,304,এটি বিআইএস প্রত্যয়িত এবং নিরাপদ বলে দাবি করা হয়।,Positive,bn,49
4,295,6 mAh બેટરી અને લાંબા સમય સુધી ચાલતી નથી.,Negative,gu,41


In [7]:
# ── Label balance per language ─────────────────────────────────────────────────
print("=== Label balance per language ===")
pivot = train_df.groupby(["language", "label"]).size().unstack(fill_value=0)
pivot["total"] = pivot.sum(axis=1)
if "Positive" in pivot.columns and "Negative" in pivot.columns:
    pivot["pos_pct"] = (pivot["Positive"] / pivot["total"] * 100).round(1)
display(pivot)

=== Label balance per language ===


label,Negative,Positive,total,pos_pct
language,,,,
as,35,36,71,50.7
bd,35,36,71,50.7
bn,31,34,65,52.3
gu,34,35,69,50.7
hi,35,39,74,52.7
kn,33,33,66,50.0
ml,33,35,68,51.5
mr,36,31,67,46.3
or,35,37,72,51.4


## Step 6: Prompt Engineering

We use Gemma-3-1B-IT's native chat template (instruction-tuned format).
The training prompt wraps each sentence in a sentiment classification instruction;
the model is asked to respond with **only** `Positive` or `Negative`.

In [ ]:
FEW_SHOT_EXAMPLES = [
    {"sentence": "কর্মীদের ভাল আচরণ এবং খাবার চমৎকার ছিল।",  "language": "bn", "label": "Positive"},
    {"sentence": "খাবারের মান একদম খারাপ ছিল।",                "language": "bn", "label": "Negative"},
    {"sentence": "बहुत अच्छा अनुभव था।",                       "language": "hi", "label": "Positive"},
    {"sentence": "सेवा बहुत खराब थी।",                         "language": "hi", "label": "Negative"},
]

def build_user_message(sentence: str, language: str, include_few_shot: bool = False) -> str:
    """
    Build the user-turn content for both training and inference.
    include_few_shot: prepend 2 labelled examples to guide the model (used only at inference
                      for languages where fine-tuning data was sparse).
    """
    lang_name = LANGUAGE_MAP.get(language, language)
    header = (
        "You are a multilingual sentiment analysis expert specialising in Indian languages. "
        "Classify each text as exactly one of: Positive, Negative.\n"
    )
    if include_few_shot:
        examples = []
        for ex in FEW_SHOT_EXAMPLES[:2]:
            ex_lang = LANGUAGE_MAP.get(ex["language"], ex["language"])
            examples.append(
                f"Language: {ex_lang}\nText: {ex['sentence']}\nSentiment: {ex['label']}"
            )
        header += "\nExamples:\n" + "\n\n".join(examples) + "\n\n"

    return (
        f"{header}\n"
        f"Language: {lang_name}\n"
        f"Text: {sentence}\n\n"
        "Sentiment (respond with one word only — Positive or Negative):"
    )


def format_for_training(row, tokenizer):
    """Return the fully tokenised chat string for one training example."""
    messages = [
        {"role": "user",      "content": build_user_message(row["sentence"], row["language"])},
        {"role": "assistant", "content": row["label"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


def format_for_inference(sentence: str, language: str, tokenizer) -> str:
    """Return the prompt string (without assistant turn) for inference."""
    messages = [
        {"role": "user", "content": build_user_message(sentence, language)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


print("Prompt builder ready.")
print("Sample user message:")
print(build_user_message("بہت اچھا تجربہ تھا", "ur"))

Prompt builder ready.
Sample user message:
You are a multilingual sentiment analysis expert specialising in Indian languages. Classify each text as exactly one of: Positive, Negative.

Language: Urdu
Text: بہت اچھا تجربہ تھا

Sentiment (respond with one word only — Positive or Negative):


## Step 7: Data Preparation (Train / Validation Split)

In [ ]:

if TRAIN_ON_FULL_DATA:
    train_split = train_df.copy().reset_index(drop=True)
    val_split   = pd.DataFrame(columns=train_df.columns)  # empty — no validation
    print(f"TRAIN_ON_FULL_DATA = True")
    print(f"Training on ALL {len(train_split)} samples (no validation holdout).")
else:
    train_split, val_split = train_test_split(
        train_df,
        test_size=VAL_SPLIT,
        random_state=SEED,
        stratify=train_df["label"],
    )
    train_split = train_split.reset_index(drop=True)
    val_split   = val_split.reset_index(drop=True)
    print(f"Training   : {len(train_split)} samples")
    print(f"Validation : {len(val_split)} samples")
    print(f"Train label dist:\n{train_split['label'].value_counts()}")
    print(f"Val   label dist:\n{val_split['label'].value_counts()}")


TRAIN_ON_FULL_DATA = True
Training on ALL 900 samples (no validation holdout).


## Step 8: Load Tokenizer & Model (QLoRA)

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────────────
print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN or None,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Vocab size       : {tokenizer.vocab_size:,}")
print(f"Pad token        : {tokenizer.pad_token!r}")
print(f"EOS token        : {tokenizer.eos_token!r}")

sample_prompt = format_for_training(
    {"sentence": "अच्छा था", "language": "hi", "label": "Positive"}, tokenizer
)
print("\nSample training prompt (first 300 chars):")
print(sample_prompt[:300])

Loading tokenizer: google/gemma-3-1b-it


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Vocab size       : 262,144
Pad token        : '<pad>'
EOS token        : '<eos>'

Sample training prompt (first 300 chars):
<bos><start_of_turn>user
You are a multilingual sentiment analysis expert specialising in Indian languages. Classify each text as exactly one of: Positive, Negative.

Language: Hindi
Text: अच्छा था

Sentiment (respond with one word only — Positive or Negative):<end_of_turn>
<start_of_turn>model
Posi


In [ ]:

# ── 4-bit quantization config (bitsandbytes) ───────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normal Float 4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # ~0.4 bit additional savings
)

# ── Load base model ────────────────────────────────────────────────────────────
print(f"Loading model: {MODEL_ID}  (this may take a few minutes)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN or None,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False          # Required for gradient checkpointing
model.config.pretraining_tp = 1        # Disable tensor parallelism for stability

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

total_params    = sum(p.numel() for p in model.parameters())
print(f"Base model loaded: {total_params / 1e9:.2f}B parameters")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model: google/gemma-3-1b-it  (this may take a few minutes)


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Base model loaded: 0.65B parameters


## Step 9: Apply LoRA Adapters

In [12]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA adapters applied successfully.")

trainable params: 26,091,520 || all params: 1,025,977,472 || trainable%: 2.5431

LoRA adapters applied successfully.


## Step 10: Build Dataset for SFTTrainer

In [13]:

# ── Guard: ensure train_split exists (split cell must have run first) ──────────
if "train_split" not in globals():
    raise RuntimeError(
        "train_split is not defined.\n"
        "Please run Step 7 (Data Preparation / Train-Val Split) cell first, "
        "then re-run this cell."
    )

# ── Format all training examples using the Gemma chat template ────────────────
def build_hf_dataset(df, tokenizer):
    records = []
    for _, row in df.iterrows():
        text_str = format_for_training(row, tokenizer)
        records.append({"text": text_str})
    return Dataset.from_list(records)

train_dataset = build_hf_dataset(train_split, tokenizer)
print(f"Dataset size: {len(train_dataset)}")
print("\nSample record (first 400 chars):")
print(train_dataset[0]["text"][:400])


Dataset size: 900

Sample record (first 400 chars):
<bos><start_of_turn>user
You are a multilingual sentiment analysis expert specialising in Indian languages. Classify each text as exactly one of: Positive, Negative.

Language: Punjabi
Text: ਇਹ ਫਿਲਮ ਇੱਕ ਬੇਹਤਰੀਨ ਕਹਾਣੀ ਸੁਣਾਉਣ ਦਾ ਸਭ ਤੋਂ ਵਧੀਆ ਉਦਾਹਰਣ ਹੈ, ਪਰ ਇਹ ਸਭ ਨਕਾਰਾਤਮਕ ਅਰਥਾਂ ਵਿੱਚ ਕੀਤਾ ਗਿਆ ਹੈ। ਤੁਹਾਨੂੰ ਹਰ ਦ੍ਰਿਸ਼ ਨੂੰ ਸਮਝਣ ਲਈ ਸਮਾਂ ਕੱਢਣਾ ਹੋਵੇਗਾ!

Sentiment (respond with one word only — Positive or Negati


In [ ]:

# ── Completion-only data collator ──────────────────────────────────────────────
DataCollatorForCompletionOnlyLM = globals().get("DataCollatorForCompletionOnlyLM", None)

# One more attempt if still None — covers the case of a freshly restarted kernel
# where only THIS cell is being re-run.
if DataCollatorForCompletionOnlyLM is None:
    import importlib
    for _mod in ["trl", "trl.trainer", "trl.trainer.utils", "trl.data_utils"]:
        try:
            _m  = importlib.import_module(_mod)
            _cls = getattr(_m, "DataCollatorForCompletionOnlyLM", None)
            if _cls is not None:
                DataCollatorForCompletionOnlyLM = _cls
                print(f"[retry] DataCollatorForCompletionOnlyLM found in: {_mod}")
                break
        except Exception:
            continue

RESPONSE_TEMPLATE = "<start_of_turn>model\n"
collator = None

if DataCollatorForCompletionOnlyLM is not None:
    response_template_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
    print(f"Response template tokens : {response_template_ids}")
    print(f"Response template decoded: {tokenizer.decode(response_template_ids)!r}")

    try:
        collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template_ids,
            tokenizer=tokenizer,
        )
        print("Completion-only collator ready (token-ID mode).")
    except Exception as e1:
        print(f"Token-ID mode failed ({e1}), trying string mode...")
        try:
            collator = DataCollatorForCompletionOnlyLM(
                response_template=RESPONSE_TEMPLATE,
                tokenizer=tokenizer,
            )
            print("Completion-only collator ready (string mode).")
        except Exception as e2:
            print(f"String mode also failed ({e2}). Falling back to standard collator.")
            collator = None
else:
    print("DataCollatorForCompletionOnlyLM unavailable — using SFTTrainer default collator.")

if collator is None:
    print("Using SFTTrainer default collator (full-sequence loss). Training will still work.")

print(f"\nCollator: {type(collator).__name__ if collator is not None else 'Default (None)'}")


DataCollatorForCompletionOnlyLM unavailable — using SFTTrainer default collator.
Using SFTTrainer default collator (full-sequence loss). Training will still work.

Collator: Default (None)


## Step 11: Fine-tuning with SFTTrainer

In [15]:

# ── Guard: ensure all prerequisite variables exist ─────────────────────────────
_missing = [v for v in ["model", "tokenizer", "train_dataset", "collator", "USE_SFT_CONFIG", "SEED"]
            if v not in globals()]
if _missing:
    raise RuntimeError(
        f"The following required variables are not defined: {_missing}\n"
        "Please run all preceding cells in order (Steps 1–10) before this one."
    )

import inspect
os.makedirs(OUTPUT_DIR, exist_ok=True)

if USE_SFT_CONFIG:
    from trl import SFTConfig

    # Detect every parameter the installed SFTConfig accepts — varies by TRL version
    _sft_params = set(inspect.signature(SFTConfig.__init__).parameters.keys())

    # ── Core args: present in every version ────────────────────────────────────
    _config_kwargs = dict(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
        optim="paged_adamw_8bit",
        dataloader_pin_memory=False,
    )

    # warmup_ratio vs warmup_steps — newer TRL deprecates warmup_ratio
    if "warmup_steps" in _sft_params:
        # compute warmup_steps from ratio and total steps
        _steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
        _total_steps = _steps_per_epoch * NUM_EPOCHS
        _warmup_steps = max(1, int(_total_steps * WARMUP_RATIO))
        _config_kwargs["warmup_steps"] = _warmup_steps
        print(f"  SFTConfig ← warmup_steps = {_warmup_steps} (from ratio {WARMUP_RATIO})")
    elif "warmup_ratio" in _sft_params:
        _config_kwargs["warmup_ratio"] = WARMUP_RATIO
        print(f"  SFTConfig ← warmup_ratio = {WARMUP_RATIO}")

    # ── Version-gated args: only add if SFTConfig accepts them ─────────────────
    _optional = {
        "max_seq_length":     MAX_SEQ_LEN,   # TRL 0.12 – 0.14
        "max_length":         MAX_SEQ_LEN,   # TRL 0.15+
        "dataset_text_field": "text",
        "packing":            False,
        "group_by_length":    True,           # Removed in newer TRL
    }
    for _k, _v in _optional.items():
        if _k in _sft_params:
            _config_kwargs[_k] = _v
            print(f"  SFTConfig ← {_k} = {_v!r}")

    training_args = SFTConfig(**_config_kwargs)

    # ── SFTTrainer kwargs ──────────────────────────────────────────────────────
    _trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator,
        processing_class=tokenizer,   # Prevents SFTTrainer auto-loading image processor
    )

    # If neither max_length nor max_seq_length ended up in SFTConfig, try SFTTrainer
    _seq_len_in_config = "max_seq_length" in _sft_params or "max_length" in _sft_params
    if not _seq_len_in_config:
        _trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
        if "max_seq_length" in _trainer_params:
            _trainer_kwargs["max_seq_length"] = MAX_SEQ_LEN
            print(f"  SFTTrainer ← max_seq_length = {MAX_SEQ_LEN}")
        else:
            tokenizer.model_max_length = MAX_SEQ_LEN
            print(f"  tokenizer.model_max_length = {MAX_SEQ_LEN}")

    # dataset_text_field on SFTTrainer if not in SFTConfig
    if "dataset_text_field" not in _sft_params:
        _trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
        if "dataset_text_field" in _trainer_params:
            _trainer_kwargs["dataset_text_field"] = "text"
            print("  SFTTrainer ← dataset_text_field = 'text'")

    trainer = SFTTrainer(**_trainer_kwargs)

else:
    # ── Older TRL without SFTConfig ────────────────────────────────────────────
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        fp16=False,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
        optim="paged_adamw_8bit",
        group_by_length=True,
        dataloader_pin_memory=False,
    )
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator,
        processing_class=tokenizer,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field="text",
    )

print("\nSFTTrainer configured successfully.")
print(f"  Collator         : {type(collator).__name__ if collator is not None else 'Default (None)'}")
print(f"  SFTConfig        : {USE_SFT_CONFIG}")
print(f"  processing_class : tokenizer ({type(tokenizer).__name__})")
print("Ready to train — run the next cell to start.")


  SFTConfig ← warmup_steps = 33 (from ratio 0.1)
  SFTConfig ← max_length = 256
  SFTConfig ← dataset_text_field = 'text'
  SFTConfig ← packing = False


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]


SFTTrainer configured successfully.
  Collator         : Default (None)
  SFTConfig        : True
  processing_class : tokenizer (GemmaTokenizer)
Ready to train — run the next cell to start.


In [16]:
# ── Training ───────────────────────────────────────────────────────────────────
train_result = trainer.train()

print("\n=== Training Complete ===")
print(f"Total steps   : {train_result.global_step}")
print(f"Training loss : {train_result.training_loss:.4f}")
print(f"Runtime (s)   : {train_result.metrics.get('train_runtime', 'N/A'):.0f}")

# Save adapter weights and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel + tokenizer saved to: {OUTPUT_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss
10,6.089789
20,3.674098
30,2.582835
40,2.356788
50,2.137357
60,2.006067
70,1.808253
80,1.761746
90,1.709786
100,1.760569



=== Training Complete ===
Total steps   : 342
Training loss : 1.6601
Runtime (s)   : 2938

Model + tokenizer saved to: /kaggle/working/gemma3-sentiment


## Step 12: Inference Utilities

In [17]:
# ── Prepare model for inference ────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

model.config.use_cache = True   # Re-enable KV cache for fast generation
model.eval()

# Left-padding is better for batch generation (pad on left, generate on right)
tokenizer.padding_side = "left"

# ── Pre-compute label token IDs ────────────────────────────────────────────────
# We will compare logits of the first generated token to decide the class.
# Checking both with and without a leading space to be robust.
def get_token_candidates(tokenizer, word: str):
    """Return all plausible token IDs for a word (with/without leading space)."""
    ids = set()
    for variant in [word, " " + word, "▁" + word]:
        enc = tokenizer.encode(variant, add_special_tokens=False)
        if enc:
            ids.add(enc[0])   # Only first subword matters
    return list(ids)

POSITIVE_IDS = get_token_candidates(tokenizer, "Positive")
NEGATIVE_IDS = get_token_candidates(tokenizer, "Negative")
print(f"'Positive' candidate token IDs : {POSITIVE_IDS}")
print(f"'Negative' candidate token IDs : {NEGATIVE_IDS}")


def parse_prediction(decoded_text: str) -> str:
    """Parse the model's generated text into a clean label."""
    t = decoded_text.strip().lower()
    if "positive" in t:
        return "Positive"
    elif "negative" in t:
        return "Negative"
    elif t.startswith("pos"):
        return "Positive"
    elif t.startswith("neg"):
        return "Negative"
    else:
        return None   # Signal that logits fallback is needed


def logits_predict(model, tokenizer, prompt_text: str) -> str:
    """Fast single-sample prediction via next-token logits (no generation)."""
    inputs = tokenizer(
        prompt_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN
    ).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1, :]  # Last-position logits
    pos_score = max(logits[i].item() for i in POSITIVE_IDS) if POSITIVE_IDS else float("-inf")
    neg_score = max(logits[i].item() for i in NEGATIVE_IDS) if NEGATIVE_IDS else float("-inf")
    return "Positive" if pos_score >= neg_score else "Negative"


print("Inference utilities ready.")

'Positive' candidate token IDs : [44771, 48084]
'Negative' candidate token IDs : [63702, 57647]
Inference utilities ready.


In [18]:
def predict_dataframe(
    df: pd.DataFrame,
    model,
    tokenizer,
    batch_size: int = INFER_BATCH_SIZE,
) -> list:
    """
    Generate sentiment predictions for all rows in df.
    Strategy:
      1. Batch-generate up to MAX_NEW_TOKENS.
      2. Parse the generated text (text matching).
      3. If parsing fails, fall back to single-sample logits prediction.
    """
    predictions = []
    df_reset = df.reset_index(drop=True)
    n = len(df_reset)

    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = df_reset.iloc[start:end]

        # Build prompts
        prompts = [
            format_for_inference(row["sentence"], row["language"], tokenizer)
            for _, row in batch.iterrows()
        ]

        # Tokenise with left padding
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,         # Greedy — deterministic
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for idx, (out, row) in enumerate(zip(outputs, batch.itertuples(index=False))):
            new_tokens  = out[input_len:]   # Slice off the prompt tokens
            decoded     = tokenizer.decode(new_tokens, skip_special_tokens=True)
            pred        = parse_prediction(decoded)

            if pred is None:
                # Logits fallback — reliable even when generation is ambiguous
                prompt_text = prompts[idx]
                pred = logits_predict(model, tokenizer, prompt_text)
                print(f"    [logits fallback] row {start+idx} → {pred!r} (raw: {decoded!r})")

            predictions.append(pred)

        print(f"  Processed {end}/{n} samples")

    return predictions


print("predict_dataframe() ready.")

predict_dataframe() ready.


## Step 13: Evaluate on Validation Set

In [19]:

if len(val_split) == 0:
    print("TRAIN_ON_FULL_DATA = True — validation skipped.")
    print("All 900 samples were used for training. Proceeding directly to test inference.")
else:
    print(f"Running validation on {len(val_split)} samples...\n")

    val_preds  = predict_dataframe(val_split, model, tokenizer)
    val_labels = val_split["label"].tolist()

    macro_f1 = f1_score(val_labels, val_preds, average="macro")
    print(f"\n{'='*50}")
    print(f"Validation Macro F1-Score : {macro_f1:.4f}")
    print(f"{'='*50}\n")
    print(classification_report(val_labels, val_preds, digits=4))

    val_split["pred"] = val_preds
    print("\nF1 per language:")
    for lang, grp in val_split.groupby("language"):
        if len(grp) > 0:
            lang_f1 = f1_score(grp["label"], grp["pred"], average="macro", zero_division=0)
            lang_name = LANGUAGE_MAP.get(lang, lang)
            print(f"  {lang:3s} ({lang_name:12s}): F1={lang_f1:.3f}  n={len(grp)}")


TRAIN_ON_FULL_DATA = True — validation skipped.
All 900 samples were used for training. Proceeding directly to test inference.


## Step 14: Generate Test Predictions

In [20]:
print(f"Generating predictions for {len(test_df)} test samples...\n")

test_preds = predict_dataframe(test_df, model, tokenizer)

print(f"\nPrediction distribution:")
pred_counts = pd.Series(test_preds).value_counts()
print(pred_counts)

print(f"\nSample predictions:")
for i in range(min(5, len(test_df))):
    row = test_df.iloc[i]
    lang_name = LANGUAGE_MAP.get(row["language"], row["language"])
    print(f"  [{lang_name:12s}] {str(row['sentence'])[:60]:60s} → {test_preds[i]}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating predictions for 100 test samples...

    [logits fallback] row 0 → 'Negative' (raw: ' ഫ്രഞ്ച് ഫ്രഞ്ച് ഫ്രഞ്ച് ഫ')
  Processed 8/100 samples
  Processed 16/100 samples
  Processed 24/100 samples
  Processed 32/100 samples
  Processed 40/100 samples
  Processed 48/100 samples
  Processed 56/100 samples
  Processed 64/100 samples
  Processed 72/100 samples
  Processed 80/100 samples
    [logits fallback] row 80 → 'Positive' (raw: 'ళ్లి\n')
    [logits fallback] row 81 → 'Negative' (raw: 'ళ్లి.\nSentiment (respond with one word')
    [logits fallback] row 82 → 'Negative' (raw: 'ళ్లి\n')
    [logits fallback] row 86 → 'Negative' (raw: 'ళ్లి\n')
  Processed 88/100 samples
  Processed 96/100 samples
  Processed 100/100 samples

Prediction distribution:
Positive    55
Negative    45
Name: count, dtype: int64

Sample predictions:
  [Malayalam   ] നന്നായി പരിപാലിക്കപ്പെടുന്നതും വൃത്തിയുള്ളതുമായ കുളിമുറികൾ,  → Negative
  [Malayalam   ] ക്യാമറ ലെൻസിൽ നന്നായി ചേരുന്നതിനാൽ പൊടിയിൽ നിന്ന് 

## Step 15: Create & Verify Submission File

In [21]:

# ── Build submission DataFrame ─────────────────────────────────────────────────
# IMPORTANT: sample_submission.csv uses NUMERIC labels: 1 = Positive, 0 = Negative.
# Convert our string predictions to integers before saving.
test_preds_int = [LABEL2INT[p] for p in test_preds]

submission = pd.DataFrame({
    "ID":    test_df["ID"].values,
    "label": test_preds_int,
})

submission.to_csv(SUBMIT_PATH, index=False)

# ── Validation checks ──────────────────────────────────────────────────────────
assert list(submission.columns) == ["ID", "label"], \
    f"Column mismatch: expected ['ID', 'label'], got {submission.columns.tolist()}"

assert submission["label"].isin([0, 1]).all(), \
    f"Invalid labels found: {submission['label'].unique().tolist()}"

assert len(submission) == len(test_df), \
    f"Row count mismatch: expected {len(test_df)}, got {len(submission)}"

print(f"Submission saved to: {SUBMIT_PATH}")
print(f"\nSubmission preview:")
display(submission.head(10))
print(f"\nShape        : {submission.shape}")
print(f"Label counts : 1(Positive)={submission['label'].sum()}  0(Negative)={(submission['label']==0).sum()}")
print("\n✓ All validation checks passed. Ready for Kaggle submission!")


Submission saved to: /kaggle/working/submission.csv

Submission preview:


,ID,label
0,550,0
1,397,1
2,757,1
3,407,0
4,294,1
5,672,1
6,598,0
7,839,0
8,921,0
9,554,0



Shape        : (100, 2)
Label counts : 1(Positive)=55  0(Negative)=45

✓ All validation checks passed. Ready for Kaggle submission!


## (Optional) Step 16: Zero-Shot Baseline Comparison

Run this cell **before fine-tuning** (or load the original model) to measure
how much improvement fine-tuning gave. Useful for your documentation.

In [22]:
# This cell is informational — only run it separately to compare baselines.
# The submission above already uses the fine-tuned model.

# HOW TO RUN ZERO-SHOT:
# 1. Comment out the fine-tuning cells (Steps 11)
# 2. Run this cell right after loading the tokenizer and base model.
# 3. Compare zero-shot F1 vs. fine-tuned F1.

# Expected baseline:
#   Zero-shot Macro F1  ~ 0.60-0.72
#   Fine-tuned Macro F1 ~ 0.85-0.93
print("Zero-shot baseline not measured in this run.")
print("The fine-tuned model was used for all predictions.")

Zero-shot baseline not measured in this run.
The fine-tuned model was used for all predictions.


## Summary of Approach

| Component | Choice | Rationale |
|---|---|---|
| Base model | `google/gemma-3-1b-it` | Mandatory; instruction-tuned, multilingual tokenizer |
| Quantization | 4-bit NF4 (QLoRA) | Fits in 16 GB VRAM; negligible quality loss |
| LoRA rank | 16 | Balance between capacity and overfitting on 900 samples |
| Target modules | All attention + MLP linear layers | Maximum adaptation coverage |
| Loss masking | Completion-only (`DataCollatorForCompletionOnlyLM`) | Only the label token contributes to loss |
| Optimiser | `paged_adamw_8bit` | Memory-efficient, compatible with 4-bit training |
| LR schedule | Cosine with 10 % warmup | Prevents early training instability |
| Inference | Greedy decode + logits fallback | Deterministic; robust to edge cases |
| Evaluation | Macro F1 | Matches competition metric exactly |

